# Week 5 & 6 Deliverables   

## 1. Download Data

### Samples
- [Short-read Illumina (**interleaved** paired-end FASTQ)](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2)
- [Long-read PacBio](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2)

### Reference Genome
The hg38 (or GRCh38) version of the human genome, focusing on the chromosome that contains these genes ([chromosome 10](https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz)): 
- [CYP2C8](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A95036772%2D95069497&hgsid=3290464905_o8n5jQXlACoTC3asJu2IgMaUIkFs) (regulates many drugs, including anticancer, diabetes and blood pressure drugs)
- [CYP2C9](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A94938658%2D94990091&hgsid=3290512893_ZsVBPQDxXLev3NROCdUXxmaa0Mr2) (regulates many common drugs, including warfarin / Coumadin and NSAIDs such as Advil)
- [CYP2C19](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A94762681%2D94855547&hgsid=3290512893_ZsVBPQDxXLev3NROCdUXxmaa0Mr2) (regulates… yup, many common drugs, including antiplatelet drugs, antidepressants and anti-epileptic drugs).            

|Genes | CYP2C8 | CYP2C9 | CYP2C19 |
| --- | --- | --- | ---|
| Genomic sequence | chr10:95036772-95069497 | chr10:94938658-94990091 | chr10:94762681-94855547 |
| Strand | - | + | + | 
| Genomic size | 32726 | 51434 | 92867 |

In [ ]:
!mkdir -p data

# SAMPLES
# Download Illumina and PacBio data
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2
!bunzip2 data/*.bz2

In [15]:
# REFERENCE GENOME 
# chr10 containing CYP2C genes
# Downloading important genes 

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import requests

# Output FASTA file
output_file = "data/reference_genome.fa"

# Gene coordinates (hg38, UCSC)
GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

# UCSC FASTA API template
ucsc_fasta_url = "https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chr};start={start};end={end}"

records = []

for gene, info in GENE_INFO.items():
    url = ucsc_fasta_url.format(chr=info["chr"], start=info["start"]-1, end=info["end"])
    r = requests.get(url)
    r.raise_for_status()
    seq = r.json()["dna"]

    # Create SeqRecord
    record = SeqRecord(
        Seq(seq),
        id=gene,
        description=""
    )
    records.append(record)

# Write all genes to a single FASTA
with open(output_file, "w") as f:
    SeqIO.write(records, f, "fasta")

## 2. Align Samples to Reference Genome

**Short-read Illumina (interleaved paired-end FASTQ)**  
- -x: applies multiple options at the same time 
- sr: short read alignment without slicing

**Long-read PacBio**  
- map-hifi: align PacBio high-fidelity reads to a reference genome


In [ ]:
# minimap index
!minimap2 -d data/reference_genome.mmi data/reference_genome.fa 

# Short read Illumina 
!minimap2 -ax sr data/reference_genome.mmi data/illumina.fq > data/illumina.sam 

# Long read BioPac
!minimap2 -ax map-hifi data/reference_genome.mmi data/pacbio.fq > data/pacbio.sam

In [ ]:
# Convert SAM to sorted BAM
!samtools view -bS data/illumina.sam | samtools sort -o data/illumina.bam
!samtools view -bS data/pacbio.sam | samtools sort -o data/pacbio.bam

# Index BAM for random access
!samtools index -b data/illumina.bam
!samtools index -b data/pacbio.bam

## 3. Variant Calling 
`bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz`
- mpileup part generates genotype likelihoods at each genomic position with coverage
- call part makes the actual calls to interpret above likelihoods and detect variants 
- -m switch tells the program to use the default calling method (multiallelic caller)
- -v option asks to output only variant sites
- --ploidy 2 options assume that the genome is diploid 
- -O option selects the output format -z selects the vcf.gz format 
- -o output file name  
- Do not waste computer’s time by making mpileup convert from the internal binary representation (BCF) to text (VCF), only to be immediately converted back to binary representation by call. Instead, use -Ou to work with uncompressed BCF output

`bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/illumina.norm.splitted.vcf.gz data/illumina.vcf.gz`
- -m -any options splits multialleleic into one record per ALT 
- -f option uses the reference genome for allele normalization 

In [16]:
# Index reference genome
!samtools faidx data/reference_genome.fa

In [8]:
# Call variants 
!bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz
!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/illumina.norm.splitted.vcf.gz data/illumina.vcf.gz
!bcftools index data/illumina.norm.splitted.vcf.gz

!bcftools mpileup -Ou -f data/reference_genome.fa data/pacbio.bam | bcftools call -mv --ploidy 2 -Oz -o data/pacbio.vcf.gz
!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/pacbio.norm.splitted.vcf.gz data/pacbio.vcf.gz
!bcftools index data/pacbio.norm.splitted.vcf.gz

!bcftools convert -O v data/illumina.norm.splitted.vcf.gz > data/illumina.norm.splitted.vcf
!bcftools convert -O v data/pacbio.norm.splitted.vcf.gz > data/pacbio.norm.splitted.vcf

[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	2142/9/0/41/0/0/0
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	2005/7/0/47/0/0/0


## 4. Phase Variant VCFs 

In [17]:
!extractHAIRS --bam data/illumina.bam --VCF data/illumina.norm.splitted.vcf --out data/illumina.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/illumina.fragments --VCF data/illumina.norm.splitted.vcf --output data/illumina.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio.bam --VCF data/pacbio.norm.splitted.vcf --out data/pacbio.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/pacbio.fragments --VCF data/pacbio.norm.splitted.vcf --output data/pacbio.hapcut


Extracting haplotype informative reads from bamfiles data/illumina.bam minQV 13 minMQ 20 maxIS 1000 

VCF file data/illumina.norm.splitted.vcf has 2151 variants 
adding chrom CYP2C19 to index 
adding chrom CYP2C9 to index 
adding chrom CYP2C8 to index 
vcffile data/illumina.norm.splitted.vcf chromosomes 3 hetvariants 2030 variants 2151 
detected 0 variants with two non-reference alleles, these variants will not be phased

##########################################################################################
3 chromosomes/contigs detected in input VCF file
processing each contig separately using --regions option will be more efficient for large genomes
############################################################################################

reading fasta index file data/reference_genome.fa.fai ... fasta file data/reference_genome.fa has 3 chromosomes/contigs

found match for reference contig CYP2C19 in VCF file index 
contig CYP2C19 length 92867
found match for reference contig

## 5. Variant Analysis
- After comparing the alleles between the Illumina's phased VCF and the PacBio's phased VCF, there are 783 variants shared between the VCFs, 1368 variants are unique to Illumina, and 1229 variants are unique to PacBio.

Sampled variants that are not common between two technologies:
1. CYP2C19:54675 T->C  GT=T|C DP=258 PQ=100 (Illumina only)

2. CYP2C9:18070 A->G  GT=A|G DP=277 PQ=100 (Illumina only)

3. CYP2C8:9977 TAAAA->T  GT=TAAAA/T DP=162 PQ=None (PacBio only)


- Select 2-3 variants that are not common (if any) and check which technology supports this variant. Open both BAM files in IGV and take a screenshot of each problematic discordant location. What can you deduce from these screenshots—are these variants sequencing-related artifacts or are they indeed true variants? Do this analysis for every gene.

- Expected output: Jupyter cell(s) with IGV screenshots and a discussion.

In [18]:
import pysam 

# Phased VCFs
illumina_VCF = "data/illumina.hapcut.phased.VCF"
pacbio_VCF = "data/pacbio.hapcut.phased.VCF"

# Sorted BAM
illumina_BAM = "data/illumina.bam" 
pacbio_BAM = "data/pacbio.bam"

# Reference genome
ref = "data/reference_genome.fa"

def read_variants(vcf_path):
    vcf = pysam.VariantFile(vcf_path)
    variants = {}

    for rec in vcf.fetch():
        for alt in rec.alts:
            key = (rec.chrom, rec.pos, rec.ref, alt)

            # Genotype
            samples = list(rec.samples)
            gt = None
            if samples:
                s = rec.samples[samples[0]]
                gt_idx = s.get("GT")
                if gt_idx:
                    alleles = [rec.alleles[i] if i is not None else "." for i in gt_idx]
                    gt = "|".join(alleles) if s.phased else "/".join(alleles)
            
            # Depth (DP)
            dp = None
            if samples and "DP" in rec.samples[samples[0]]:
                dp = rec.samples[samples[0]]["DP"]
            elif "DP" in rec.info:
                dp = rec.info["DP"]
            
            # Phaseing quality
            pq = None
            if samples and "PQ" in rec.samples[samples[0]]:
                pq = rec.samples[samples[0]]["PQ"]

            variants[key] = {
                "CHROM": rec.chrom,
                "POS": rec.pos,
                "REF": rec.ref,
                "ALT": alt,
                "GT": gt,
                "DP": dp,
                "PQ": pq,
            }
    return variants
        
illumina_variants = read_variants(illumina_VCF)
pacbio_variants = read_variants(pacbio_VCF)

illumina_keys = set(illumina_variants.keys())
pacbio_keys = set(pacbio_variants.keys())

# Compare
shared_variants = illumina_keys & pacbio_keys
illumina_only = illumina_keys - pacbio_keys
pacbio_only = pacbio_keys - illumina_keys

print(f"Shared variants: {len(shared_variants)}")
print(f"Unique to Illumina: {len(illumina_only)}")
print(f"Unique to PacBio:  {len(pacbio_only)}\n")

Shared variants: 783
Unique to Illumina: 1368
Unique to PacBio:  1229



In [19]:
# Choose variants to inspect
def pick_vars(keys, vars, n=3):
    keys = list(keys)
    def score(k):
        info = vars[k]
        dp = info.get("DP") or 0
        pq = info.get("PQ") or 0
        return (dp, pq)
    keys_sorted = sorted(keys, key=score, reverse=True)
    return keys_sorted[:n]

pick_illumina = pick_vars(illumina_only, illumina_variants, n=3)
pick_pacbio = pick_vars(pacbio_only, pacbio_variants, n=3)

print("\nExample discordant variants (Illumina-only):")
for k in pick_illumina:
    info = illumina_variants[k]
    print(f"{info['CHROM']}:{info['POS']} {info['REF']}->{info['ALT']}  GT={info['GT']} DP={info['DP']} PQ={info['PQ']}")

print("\nExample discordant variants (PacBio-only):")
for k in pick_pacbio:
    info = pacbio_variants[k]
    print(f"{info['CHROM']}:{info['POS']} {info['REF']}->{info['ALT']}  GT={info['GT']} DP={info['DP']} PQ={info['PQ']}")


Example discordant variants (Illumina-only):
CYP2C9:18070 A->G  GT=A|G DP=277 PQ=100
CYP2C19:54675 T->C  GT=T|C DP=258 PQ=100
CYP2C9:16508 C->T  GT=C|T DP=257 PQ=100

Example discordant variants (PacBio-only):
CYP2C8:9977 TAAAA->T  GT=TAAAA/T DP=162 PQ=None
CYP2C8:9977 TAAA->T  GT=T/TAAA DP=162 PQ=None
CYP2C8:32291 CAA->C  GT=CAA/C DP=133 PQ=None


In [ ]:
# generate_igv_batch.py
import os

# Output batch file
batch_file = "igv_commands.txt"

# Snapshot directory
snapshot_dir = "snapshots"
os.makedirs(snapshot_dir, exist_ok=True)

# BAM files
bams = ["data/illumina.bam", "data/pacbio.bam"]

# Reference genome
reference_fasta = "data/reference_genome.fa"

# Discordant variants: (gene, position)
discordant_variants = [("CYP2C19", 54675), ("CYP2C9", 18070), ("CYP2C8", 9977)]

with open(batch_file, "w") as f:
    f.write("new\n")
    f.write(f"genome {reference_fasta}\n")
    
    for bam in bams:
        f.write(f"load {bam}\n")
    
    f.write(f"snapshotDirectory {snapshot_dir}\n")
    
    # Go to each variant and take a snapshot
    for gene, pos in discordant_variants:
        start = pos - 10
        end = pos + 10
        f.write(f"goto {gene}:{start}-{end}\n")
        f.write("sort base\n")  # optional: tidy view around SNP
        f.write("collapse\n")   # optional: collapse reads per sample
        f.write(f"snapshot {gene}_{pos}.png\n")
    
    f.write("exit\n")

IGV batch file generated: igv_commands.txt
Snapshots will be saved in: snapshots/


In [ ]:
!igv -b igv_commands.txt

In [ ]:
from IPython.display import HTML

html = """
<h3>IGV Snapshots for Discordant Variants</h3>
<div style='display:flex; gap:20px; flex-wrap:wrap;'>
  <figure>
    <img src='snapshots/CYP2C19_54675.png' width='800'>
    <figcaption><b>CYP2C19 (54675)</b> — Illumina-only variant</figcaption>
  </figure>
  <figure>
    <img src='snapshots/CYP2C9_18070.png' width='800'>
    <figcaption><b>CYP2C9 (18070)</b> — Illumina-only variant</figcaption>
  </figure>
  <figure>
    <img src='snapshots/CYP2C8_9977.png' width='800'>
    <figcaption><b>CYP2C8 (9977)</b> — PacBio-only variant</figcaption>
  </figure>
</div>
"""
display(HTML(html))

## 6. Star-Allele Calls
- Can you figure out the star-allele for each gene of interest? The star-allele database can be found in PharmVar; see this for CYP2C19. Your answer should be something like CYP2C19*12 because X, Y and Z. This step does not have to be automated, but should be at least explained in the notebook.
    - Hint: use phased data!

- Expected output: Jupyter cell(s) with discussion (and code, if you want to do it that way).